# Stage 1 — EDA da base Bank Marketing

**Datathon 7MLET — Grupo 74.** Notebook reprodutível de Análise Exploratória sobre a base processada (sem vazamento).

> Roda offline com o *facsimile* determinístico ou com a base Kaggle real (se baixada — ver `data/kaggle/README.md`).

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, matplotlib.pyplot as plt
from adaptive_offers.data.preprocessing import build_processed, load_processed
from adaptive_offers.data.quality import quality_report, target_rate_by
pd.set_option("display.width", 120)

## 1. Carregar / construir a base processada
O build registra proveniência (fonte, versão, licença) em `data/processed/provenance.json`.

In [ ]:
df, prov, path = build_processed(n_rows=20000)
print("source:", prov.source, "| version:", prov.version, "| license:", prov.license)
df.head()

## 2. Perfil de qualidade
Missingness, cardinalidade, duplicatas e balanceamento do alvo.

In [ ]:
rep = quality_report(df)
print("rows:", rep["n_rows"], "cols:", rep["n_cols"], "duplicates:", rep["n_duplicates"])
print("target:", rep["target"])

## 3. Balanceamento do alvo `subscribed`
O forte desbalanceamento justifica métricas além de acurácia (uplift/regret) na Etapa 4.

In [ ]:
ax = df["subscribed"].value_counts().sort_index().plot(kind="bar")
ax.set_title("Distribuição do alvo (0=não, 1=assinou)"); ax.set_xlabel("subscribed"); plt.tight_layout(); plt.show()

## 4. Drivers de conversão
Taxa de assinatura por `poutcome` e por `contact` — sinais que alimentam o contexto do bandit (Etapa 3).

In [ ]:
print(target_rate_by(df, "poutcome").to_string())
print()
print(target_rate_by(df, "contact").to_string())

In [ ]:
g = target_rate_by(df, "poutcome")
ax = g["subscription_rate"].plot(kind="barh")
ax.set_title("Taxa de assinatura por poutcome"); plt.tight_layout(); plt.show()

## 5. Idade vs. conversão
Clientes nos extremos etários (estudantes/aposentados) tendem a converter mais.

In [ ]:
df["age_band"] = pd.cut(df["age"], bins=[17,25,35,45,55,65,100], labels=["18-25","26-35","36-45","46-55","56-65","65+"])
print(target_rate_by(df, "age_band").to_string())

## 6. Conclusões
- Sem vazamento (`duration` removida; `pdays` tratado).
- Alvo desbalanceado (~5% facsimile / ~11% real).
- Drivers: `poutcome=success`, canal `cellular`, juros baixos, extremos etários.
- Esses sinais definem o **vetor de contexto** usado pelo LinUCB na decisão de oferta.

Ver `reports/eda-quality-report.md` para o relatório consolidado.